# Figure 2 | Three study sites and survey layouts

Run the cells in order from the repository root or `examples/`. Panels are saved to `examples/Figure/fig2/` at 400 dpi. The source field data and saved case results must be obtained separately. The assembled manuscript figure is not generated here.

Panels a–d locate the three field cases. The shared method legend is also exported for assembly.

## Prepare source geometry and drawing functions

In [ ]:
"""Build Figure 2 from the saved survey and model data."""

import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Polygon, Rectangle
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
from PIL import Image
from pyproj import Transformer
import rasterio
from rasterio.windows import from_bounds
from scipy.spatial import cKDTree
from scipy.interpolate import griddata


PALETTE = {
    "gravity": "#7A5195",
    "tem": "#E17C05",
    "mt": "#008585",
    "aem": "#E17C05",
    "ert": "#D1495B",
    "srt": "#6F4E7C",
    "sp": "#5D6D7E",
    "tdem": "#E17C05",
    "well": "#1F1F1F",
    "domain": "#4D4D4D",
    "utah": "#C24E3D",
    "cedar": "#2B6F9F",
    "llano": "#3A8C6E",
}

# Standard cartographic earth palette.
TERRAIN_CMAP = mpl.colormaps["terrain"].copy()
TERRAIN_LIMITS_M = {
    "utah": (1200.0, 3000.0),
    "cedar": (210.0, 300.0),
    "llano": (285.0, 310.0),
}


def _set_style() -> None:
    mpl.rcParams.update(
        {
            "font.family": "Arial",
            "font.size": 7.0,
            "axes.titlesize": 8.2,
            "axes.labelsize": 7.2,
            "xtick.labelsize": 6.4,
            "ytick.labelsize": 6.4,
            "legend.fontsize": 5.7,
            "axes.linewidth": 0.65,
            "xtick.major.width": 0.55,
            "ytick.major.width": 0.55,
            "xtick.major.size": 2.7,
            "ytick.major.size": 2.7,
            "svg.fonttype": "none",
            "pdf.fonttype": 42,
            "savefig.facecolor": "white",
            "figure.facecolor": "white",
        }
    )


def _find_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "Result").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a parent containing data/ and Result/.")


def _read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def _bounds_xy(points: np.ndarray, pad_fraction: float = 0.04) -> tuple[float, float, float, float]:
    points = np.asarray(points, dtype=float)
    xmin, ymin = np.nanmin(points[:, :2], axis=0)
    xmax, ymax = np.nanmax(points[:, :2], axis=0)
    dx = max(xmax - xmin, 1.0)
    dy = max(ymax - ymin, 1.0)
    return (
        xmin - dx * pad_fraction,
        xmax + dx * pad_fraction,
        ymin - dy * pad_fraction,
        ymax + dy * pad_fraction,
    )


def _balanced_bounds(
    bounds_m: tuple[float, float, float, float], box_height_over_width: float = 1.03
) -> tuple[float, float, float, float]:
    """Expand, never crop, bounds so equal-scale map panels share one box shape."""
    xmin, xmax, ymin, ymax = bounds_m
    width = xmax - xmin
    height = ymax - ymin
    target_height = width * box_height_over_width
    if height < target_height:
        extra = (target_height - height) / 2
        ymin, ymax = ymin - extra, ymax + extra
    else:
        target_width = height / box_height_over_width
        extra = (target_width - width) / 2
        xmin, xmax = xmin - extra, xmax + extra
    return xmin, xmax, ymin, ymax


def _domain_rectangle(ax: plt.Axes, bounds_m: tuple[float, float, float, float]) -> None:
    xmin, xmax, ymin, ymax = bounds_m
    ax.add_patch(
        Rectangle(
            (xmin / 1000, ymin / 1000),
            (xmax - xmin) / 1000,
            (ymax - ymin) / 1000,
            fill=False,
            edgecolor=PALETTE["domain"],
            linewidth=0.9,
            linestyle=(0, (3, 2)),
            zorder=7,
        )
    )


def _panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(
        0.012,
        0.985,
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9.0,
        fontweight="bold",
        color="#111111",
        zorder=30,
    )


def _north_arrow(ax: plt.Axes, x: float = 0.94, y: float = 0.91) -> None:
    ax.annotate(
        "N",
        xy=(x, y),
        xytext=(x, y - 0.105),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=6.4,
        fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", color="#222222", lw=0.75),
        zorder=25,
    )


def _scale_bar(
    ax: plt.Axes,
    length_km: float,
    label: str | None = None,
    location: tuple[float, float] = (0.075, 0.075),
    segments: int = 4,
) -> None:
    """Draw a compact alternating-block cartographic scale bar in map units."""
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    width, height = x1 - x0, y1 - y0
    x = x0 + location[0] * width
    y = y0 + location[1] * height
    bar_h = 0.010 * height
    segment_width = length_km / segments
    for index in range(segments):
        ax.add_patch(
            Rectangle(
                (x + index * segment_width, y),
                segment_width,
                bar_h,
                facecolor="#242424" if index % 2 == 0 else "#F7F4ED",
                edgecolor="#242424",
                linewidth=0.50,
                zorder=25,
            )
        )
    ax.add_patch(
        Rectangle((x, y), length_km, bar_h, fill=False, edgecolor="#242424", linewidth=0.65, zorder=26)
    )
    ax.text(
        x + length_km / 2,
        y + 0.024 * height,
        label or f"{length_km:g} km",
        ha="center",
        va="bottom",
        fontsize=5.8,
        fontweight="bold",
        color="#202020",
        zorder=27,
    )


def _format_local_map(ax: plt.Axes, show_ylabel: bool = True) -> None:
    ax.set_aspect("equal", adjustable="box")
    ax.set_box_aspect(1.03)
    ax.set_xlabel("Easting (km)", labelpad=3, fontsize=10.2, fontweight="bold")
    ax.set_ylabel("Northing (km)" if show_ylabel else "", labelpad=3, fontsize=10.2, fontweight="bold")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.tick_params(direction="out", top=False, right=False, pad=2, labelsize=9.4)
    for label in (*ax.get_xticklabels(), *ax.get_yticklabels()):
        label.set_fontweight("bold")
    for spine in ax.spines.values():
        spine.set_color("#555555")


def _terrain_relief(
    array: np.ndarray, dx: float, dy: float, limits_m: tuple[float, float] | None = None
) -> np.ndarray:
    data = np.asarray(array, dtype=float)
    if np.ma.isMaskedArray(array):
        data = np.ma.filled(array, np.nan)
    if np.isnan(data).any():
        fill = np.nanmedian(data)
        data = np.where(np.isfinite(data), data, fill)
    vmin, vmax = limits_m if limits_m is not None else np.nanpercentile(data, [2, 98])
    if np.isclose(vmin, vmax):
        vmin, vmax = float(np.nanmin(data)), float(np.nanmax(data))
    return LightSource(azdeg=315, altdeg=38).shade(
        data,
        cmap=TERRAIN_CMAP,
        norm=Normalize(vmin=vmin, vmax=vmax, clip=True),
        blend_mode="soft",
        vert_exag=1.35,
        dx=dx,
        dy=dy,
        fraction=0.52,
    )


def _crop_dem(path: Path, bounds_m: tuple[float, float, float, float], max_pixels: int = 650):
    xmin, xmax, ymin, ymax = bounds_m
    with rasterio.open(path) as src:
        win = from_bounds(xmin, ymin, xmax, ymax, src.transform).round_offsets().round_lengths()
        win = win.intersection(rasterio.windows.Window(0, 0, src.width, src.height))
        scale = max(win.width / max_pixels, win.height / max_pixels, 1)
        out_h = max(2, int(win.height / scale))
        out_w = max(2, int(win.width / scale))
        dem = src.read(1, window=win, out_shape=(out_h, out_w), masked=True)
        left, bottom, right, top = rasterio.windows.bounds(win, src.transform)
    return dem, (left / 1000, right / 1000, bottom / 1000, top / 1000)


def _load_data(root: Path) -> dict:
    result = root / "Result"
    data = root / "data"

    # Utah FORGE
    u_state = _read_json(result / "Utah-FORGE" / "case_state.json")
    gravity = pd.read_csv(root / u_state["observations_path"])
    tem_path = result / "Utah-FORGE" / "runs" / u_state["tem_run_id"] / "stitched_conductivity_section.npz"
    tem_npz = np.load(tem_path)
    tem_xy = tem_npz["receiver_locations_m"][:, :2]
    mt_path = result / "Utah-FORGE" / "runs" / u_state["mt_run_id"] / "stitched_conductivity_section.npz"
    mt_npz = np.load(mt_path)
    mt_xy = np.c_[mt_npz["easting_m"], mt_npz["northing_m"]]
    # Final geological-model volume (NAD83 / UTM zone 12N).
    u_domain = (331_900.0, 340_150.0, 4_259_925.0, 4_265_150.0)
    # Keep the final model domain fixed.
    support_buffer_m = 2_000.0
    u_support_bounds = (
        u_domain[0] - support_buffer_m,
        u_domain[1] + support_buffer_m,
        u_domain[2] - support_buffer_m,
        u_domain[3] + support_buffer_m,
    )
    u_plot_bounds = _balanced_bounds(u_support_bounds, box_height_over_width=1.03)
    mt_local = mt_xy[
        (mt_xy[:, 0] >= u_plot_bounds[0])
        & (mt_xy[:, 0] <= u_plot_bounds[1])
        & (mt_xy[:, 1] >= u_plot_bounds[2])
        & (mt_xy[:, 1] <= u_plot_bounds[3])
    ]
    to_u = Transformer.from_crs(4326, 26912, always_xy=True)
    forge_well = np.array(to_u.transform(-112.896407, 38.504013))

    # Cedar Rapids
    c_state = _read_json(result / "Cedar-Rapids" / "case_state.json")
    aem_path = result / "Cedar-Rapids" / "runs" / c_state["aem_run_id"] / "stitched_conductivity_section.npz"
    aem_npz = np.load(aem_path)
    aem_xy = aem_npz["receiver_locations_m"][:, :2]
    aem_ids = aem_npz["sounding_ids"].astype(str)
    aem_lines = np.array([s.split("-")[0] for s in aem_ids])
    raw_dtm = pd.read_csv(
        data / "case2_Cedarrapids" / "aem" / "inversion_EM1DFM.csv",
        usecols=["X_NAD83UTM15N_M", "Y_NAD83UTM15N_M", "DTM_HEM"],
    )
    tree = cKDTree(raw_dtm[["X_NAD83UTM15N_M", "Y_NAD83UTM15N_M"]].to_numpy())
    _, nearest = tree.query(aem_xy, k=1)
    aem_z = raw_dtm["DTM_HEM"].to_numpy()[nearest]
    cedar_model = np.load(root / c_state["geological_model_path"])["cell_centres_m"]
    c_domain = _bounds_xy(cedar_model[:, :2], pad_fraction=0)
    to_c = Transformer.from_crs(4326, 26915, always_xy=True)
    ert_endpoints_ll = {
        "Ellis CW1": ((-91.7340, 41.9984), (-91.7361, 42.0024)),
        "SVP CW6": ((-91.7417, 42.0076), (-91.7448, 42.0085)),
        "SVP S10": ((-91.7313, 41.9985), (-91.7284, 41.9982)),
        "SVP Demo": ((-91.7342, 42.0054), (-91.7310, 42.0049)),
    }
    cedar_ert = {
        name: np.array([to_c.transform(*p0), to_c.transform(*p1)])
        for name, (p0, p1) in ert_endpoints_ll.items()
    }
    cedar_wells_ll = {
        "GX-1": (-91.74250, 42.00861),
        "GX-2": (-91.73445, 41.99833),
        "CRM-4A": (-91.73139, 41.99806),
        "CRM-6A": (-91.73139, 41.99833),
    }
    cedar_wells = {name: np.array(to_c.transform(*ll)) for name, ll in cedar_wells_ll.items()}

    # Llano
    l_state = _read_json(result / "Llano-Uplift" / "case_state.json")
    ert_table = pd.read_csv(data / "case3_Llano" / "ert" / "electrodes_xyz.txt", sep="\t")
    srt_table = pd.read_csv(data / "case3_Llano" / "srt" / "geometry.txt", sep="\t")
    sp_table = pd.read_csv(data / "case3_Llano" / "sp" / "survey.txt", sep="\t")
    to_l = Transformer.from_crs(4326, 26914, always_xy=True)

    def project_frame(frame: pd.DataFrame) -> np.ndarray:
        x, y = to_l.transform(frame["Longitude"].to_numpy(), frame["Latitude"].to_numpy())
        return np.c_[x, y]

    llano_ert = {
        int(profile): project_frame(group)
        for profile, group in ert_table.groupby("ERT_Profile_ID", sort=True)
    }
    llano_srt = {
        int(profile): project_frame(group)
        for profile, group in srt_table.groupby("SRT_Profile_ID", sort=True)
    }
    sp_xy = project_frame(sp_table)
    tdem_xy = np.asarray(l_state["tdem_import"]["location_m"][:2], dtype=float)
    tdem_radius = float(l_state["tdem_import"]["loop_radius_m"])
    # The loop is 100 by 100 m and the SP survey is gridded 0-100 m inside it with
    # latitude and longitude at every station, so an affine fit of that grid places the
    # square (the case script places its centre the same way).
    local = sp_table[["Line_x_Coordinate_meters", "Line_y_Coordinate_meters"]].to_numpy(dtype=float)
    affine, *_ = np.linalg.lstsq(np.c_[local, np.ones(len(local))], sp_xy, rcond=None)
    square = np.array([[0.0, 0.0], [100.0, 0.0], [100.0, 100.0], [0.0, 100.0]])
    tdem_loop = np.c_[square, np.ones(4)] @ affine
    llano_model = np.load(root / l_state["geological_model_path"])["cell_centres_m"]
    l_domain = _bounds_xy(llano_model[:, :2], pad_fraction=0)
    topo_xy = np.vstack([project_frame(ert_table), project_frame(srt_table), sp_xy])
    topo_z = np.r_[
        ert_table["Surface_Elevation_meters_above_NAVD88"].to_numpy(),
        srt_table["Surface_Elevation_meters_above_NAVD88"].to_numpy(),
        sp_table["Surface_Elevation_meters_above_NAVD88"].to_numpy(),
    ]

    return {
        "utah": {
            "gravity": gravity,
            "tem_xy": tem_xy,
            "mt_xy": mt_local,
            "domain": u_domain,
            "plot_bounds": u_plot_bounds,
            "dem": data / "case1_Utah FORGE" / "dem" / "forge_3dep_30m.tif",
            "well": forge_well,
        },
        "cedar": {
            "aem_xy": aem_xy,
            "aem_z": aem_z,
            "aem_lines": aem_lines,
            "ert": cedar_ert,
            "wells": cedar_wells,
            "domain": c_domain,
        },
        "llano": {
            "ert": llano_ert,
            "srt": llano_srt,
            "sp_xy": sp_xy,
            "tdem_xy": tdem_xy,
            "tdem_radius": tdem_radius,
            "tdem_loop": tdem_loop,
            "domain": l_domain,
            "topo_xy": topo_xy,
            "topo_z": topo_z,
            "dem": data / "case3_Llano" / "dem" / "llano_3dep_1m.tif",
        },
    }


def _plot_locator(ax: plt.Axes) -> None:
    """Locate the sites on a projected graticule of the contiguous US."""
    project = Transformer.from_crs(4326, 5070, always_xy=True)
    for longitude in range(-125, -65, 10):
        latitudes = np.linspace(25, 50, 160)
        xs, ys = project.transform(np.full_like(latitudes, longitude), latitudes)
        ax.plot(xs, ys, color="#D7E1E4", lw=0.6, zorder=1)
    for latitude in range(25, 51, 5):
        longitudes = np.linspace(-125, -66, 200)
        xs, ys = project.transform(longitudes, np.full_like(longitudes, latitude))
        ax.plot(xs, ys, color="#D7E1E4", lw=0.6, zorder=1)

    sites = [
        ("Utah FORGE", -112.896, 38.504),
        ("Cedar Rapids", -91.734, 42.004),
        ("Llano", -98.383, 30.649),
    ]
    for label, longitude, latitude in sites:
        x, y = project.transform(longitude, latitude)
        ax.scatter(x, y, s=38, color="#D7654F", edgecolor="white",
                   linewidth=0.9, zorder=10)
        ax.annotate(label, (x, y), xytext=(5, 5), textcoords="offset points",
                    fontsize=7.2, color="#33474F", zorder=11)

    corners_x, corners_y = project.transform(
        np.array([-125, -125, -66, -66]), np.array([25, 50, 25, 50])
    )
    xmin, xmax = float(np.min(corners_x)), float(np.max(corners_x))
    ymin, ymax = float(np.min(corners_y)), float(np.max(corners_y))
    xpad = 0.025 * (xmax - xmin)
    ypad = 0.075 * (ymax - ymin)
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + 0.025 * (ymax - ymin))
    ax.set_aspect("equal", adjustable="box")

    # The map is projected in metres, so the scale bar is geometrically valid.
    bar_length = 1_000_000.0
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    bar_x = x0 + 0.075 * (x1 - x0)
    bar_y = y0 + 0.075 * (y1 - y0)
    bar_h = 0.014 * (y1 - y0)
    segment = bar_length / 4
    for index in range(4):
        ax.add_patch(
            Rectangle(
                (bar_x + index * segment, bar_y),
                segment,
                bar_h,
                facecolor="#40565F" if index % 2 == 0 else "#D7E1E4",
                edgecolor="#40565F",
                linewidth=0.55,
                zorder=20,
            )
        )
    label_y = bar_y - 0.009 * (y1 - y0)
    ax.text(bar_x, label_y, "0", ha="center", va="top", fontsize=5.7, color="#33474F")
    ax.text(bar_x + bar_length / 2, label_y, "500", ha="center", va="top", fontsize=5.7, color="#33474F")
    ax.text(bar_x + bar_length, label_y, "1,000", ha="center", va="top", fontsize=5.7, color="#33474F")
    ax.text(
        bar_x + bar_length + 0.035 * bar_length,
        bar_y + bar_h / 2,
        "km",
        ha="left",
        va="center",
        fontsize=5.7,
        color="#33474F",
    )

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def _plot_utah(ax: plt.Axes, d: dict) -> list[Line2D]:
    bounds = d["plot_bounds"]
    dem, extent = _crop_dem(d["dem"], bounds)
    relief = _terrain_relief(dem, dx=30, dy=30, limits_m=TERRAIN_LIMITS_M["utah"])
    ax.imshow(relief, extent=extent, origin="upper", zorder=0)
    gravity = d["gravity"]
    ax.scatter(gravity["x_m"] / 1000, gravity["y_m"] / 1000, s=4.0, color=PALETTE["gravity"], alpha=0.72, linewidths=0, zorder=4)
    ax.scatter(d["tem_xy"][:, 0] / 1000, d["tem_xy"][:, 1] / 1000, s=13, marker="s", color=PALETTE["tem"], edgecolor="white", linewidth=0.35, zorder=6)
    ax.scatter(d["mt_xy"][:, 0] / 1000, d["mt_xy"][:, 1] / 1000, s=17, marker="^", color=PALETTE["mt"], edgecolor="white", linewidth=0.35, zorder=5)
    ax.scatter(d["well"][0] / 1000, d["well"][1] / 1000, s=42, marker="*", color=PALETTE["well"], edgecolor="white", linewidth=0.45, zorder=12)
    _domain_rectangle(ax, d["domain"])
    ax.set_xlim(bounds[0] / 1000, bounds[1] / 1000)
    ax.set_ylim(bounds[2] / 1000, bounds[3] / 1000)
    handles = [
        Line2D([], [], marker="o", linestyle="", color=PALETTE["gravity"], markersize=3, label=f"Gravity ({len(gravity)})"),
        Line2D([], [], marker="s", linestyle="", color=PALETTE["tem"], markersize=3.5, label=f"TEM ({len(d['tem_xy'])})"),
        Line2D([], [], marker="^", linestyle="", color=PALETTE["mt"], markersize=4, label=f"MT shown ({len(d['mt_xy'])})"),
        Line2D([], [], marker="*", linestyle="", color=PALETTE["well"], markersize=5, label="Deep well"),
        Line2D([], [], color=PALETTE["domain"], ls=(0, (3, 2)), lw=0.9, label="Model domain"),
    ]
    _scale_bar(ax, 2)
    _format_local_map(ax)
    ax.set_box_aspect((bounds[3] - bounds[2]) / (bounds[1] - bounds[0]))
    return handles


def _plot_cedar(ax: plt.Axes, d: dict) -> list[Line2D]:
    x, y = d["aem_xy"].T
    bounds = _balanced_bounds(
        _bounds_xy(np.vstack([d["aem_xy"], np.vstack(list(d["ert"].values()))]), pad_fraction=0.06)
    )
    # The supplied AEM file includes terrain elevations; no Cedar DEM is bundled.
    gx = np.linspace(bounds[0], bounds[1], 360)
    gy = np.linspace(bounds[2], bounds[3], 360)
    grid_x, grid_y = np.meshgrid(gx, gy)
    dem = griddata(d["aem_xy"], d["aem_z"], (grid_x, grid_y), method="linear")
    nearest = griddata(d["aem_xy"], d["aem_z"], (grid_x, grid_y), method="nearest")
    dem = np.where(np.isfinite(dem), dem, nearest)
    extent = (bounds[0] / 1000, bounds[1] / 1000, bounds[2] / 1000, bounds[3] / 1000)
    dx = (bounds[1] - bounds[0]) / dem.shape[1]
    dy = (bounds[3] - bounds[2]) / dem.shape[0]
    relief = _terrain_relief(dem, dx=dx, dy=dy, limits_m=TERRAIN_LIMITS_M["cedar"])
    ax.imshow(relief, extent=extent, origin="upper", zorder=0)

    for line_id in np.unique(d["aem_lines"]):
        p = d["aem_xy"][d["aem_lines"] == line_id]
        ax.plot(p[:, 0] / 1000, p[:, 1] / 1000, color=PALETTE["aem"], lw=0.65, alpha=0.82, zorder=3)
    for name, p in d["ert"].items():
        ax.plot(p[:, 0] / 1000, p[:, 1] / 1000, color="white", lw=2.8, zorder=6)
        ax.plot(p[:, 0] / 1000, p[:, 1] / 1000, color=PALETTE["ert"], lw=1.55, zorder=7)
    well_offsets = {
        "GX-1": (3, 3),
        "GX-2": (-24, -10),
        "CRM-4A": (4, -12),
        "CRM-6A": (4, 4),
    }
    for name, p in d["wells"].items():
        ax.scatter(p[0] / 1000, p[1] / 1000, marker="*", s=34, color=PALETTE["well"], edgecolor="white", linewidth=0.4, zorder=10)
    _domain_rectangle(ax, d["domain"])
    ax.set_xlim(bounds[0] / 1000, bounds[1] / 1000)
    ax.set_ylim(bounds[2] / 1000, bounds[3] / 1000)
    handles = [
        Line2D([], [], color=PALETTE["aem"], lw=1.2, label=f"AEM ({len(d['aem_xy']):,})"),
        Line2D([], [], color=PALETTE["ert"], lw=1.8, label=f"ERT sites ({len(d['ert'])})"),
        Line2D([], [], marker="*", linestyle="", color=PALETTE["well"], markersize=5, label=f"Wells ({len(d['wells'])})"),
        Line2D([], [], color=PALETTE["domain"], ls=(0, (3, 2)), lw=0.9, label="Model domain"),
    ]
    _scale_bar(ax, 1)
    _format_local_map(ax, show_ylabel=False)
    return handles


def _plot_llano(ax: plt.Axes, d: dict) -> list[Line2D]:
    xmin, xmax, ymin, ymax = d["domain"]
    centre = np.array([(xmin + xmax) / 2, (ymin + ymax) / 2])
    domain_corners = np.array([[xmin, ymin], [xmin, ymax], [xmax, ymin], [xmax, ymax]])
    tdem_extrema = d["tdem_loop"]
    shown_xy = np.vstack(
        [domain_corners, d["sp_xy"], np.vstack(list(d["ert"].values())), d["srt"][2], tdem_extrema]
    )
    half_width = np.max(np.abs(shown_xy[:, 0] - centre[0])) * 1.10
    half_height = np.max(np.abs(shown_xy[:, 1] - centre[1])) * 1.10
    if half_height < half_width * 1.03:
        half_height = half_width * 1.03
    else:
        half_width = half_height / 1.03
    bounds = (
        centre[0] - half_width,
        centre[0] + half_width,
        centre[1] - half_height,
        centre[1] + half_height,
    )
    dem, extent = _crop_dem(d["dem"], bounds, max_pixels=1200)
    dx = (extent[1] - extent[0]) * 1000 / dem.shape[1]
    dy = (extent[3] - extent[2]) * 1000 / dem.shape[0]
    relief = _terrain_relief(dem, dx=dx, dy=dy, limits_m=TERRAIN_LIMITS_M["llano"])
    ax.imshow(relief, extent=extent, origin="upper", zorder=0)
    ax.scatter(d["sp_xy"][:, 0] / 1000, d["sp_xy"][:, 1] / 1000, s=2.0, color=PALETTE["sp"], alpha=0.55, linewidths=0, zorder=3)
    for i, p in d["ert"].items():
        ax.plot(p[:, 0] / 1000, p[:, 1] / 1000, color="white", lw=3.0, zorder=5)
        ax.plot(p[:, 0] / 1000, p[:, 1] / 1000, color=PALETTE["ert"], lw=1.7, zorder=6)
    for i, p in d["srt"].items():
        if i == 1:
            continue
        ax.plot(p[:, 0] / 1000, p[:, 1] / 1000, color=PALETTE["srt"], lw=1.15, ls=(0, (3, 1.5)), zorder=6)
    # The transmitter loop as laid out: a 100 m square.
    for colour, width, order in (("white", 3.0, 8), (PALETTE["tdem"], 1.6, 9)):
        ax.add_patch(Polygon(d["tdem_loop"] / 1000, closed=True, fill=False,
                             edgecolor=colour, linewidth=width, zorder=order))
    # Where ERT2, SRT2 and the loop centre meet (0.6 m and 2.3 m apart): one column, three methods.
    ax.scatter(d["tdem_xy"][0] / 1000, d["tdem_xy"][1] / 1000, s=34, facecolor="none",
               edgecolor="#111111", linewidth=1.1, zorder=12)
    ax.scatter(d["tdem_xy"][0] / 1000, d["tdem_xy"][1] / 1000, s=58, facecolor="none",
               edgecolor="white", linewidth=0.8, zorder=11)
    _domain_rectangle(ax, d["domain"])
    ax.set_xlim(bounds[0] / 1000, bounds[1] / 1000)
    ax.set_ylim(bounds[2] / 1000, bounds[3] / 1000)
    handles = [
        Line2D([], [], marker="o", linestyle="", color=PALETTE["sp"], markersize=2.5, label=f"SP ({len(d['sp_xy'])})"),
        Line2D([], [], color=PALETTE["ert"], lw=1.8, label=f"ERT lines ({len(d['ert'])})"),
        Line2D([], [], color=PALETTE["srt"], lw=1.2, ls=(0, (3, 1.5)), label=f"SRT lines ({len(d['srt'])})"),
        Line2D([], [], marker="s", markerfacecolor="none", linestyle="", color=PALETTE["tdem"], markersize=5, label="TEM loop"),
        Line2D([], [], color=PALETTE["domain"], ls=(0, (3, 2)), lw=0.9, label="Model domain"),
    ]
    _scale_bar(ax, 0.1, "100 m", location=(0.075, 0.130))
    _format_local_map(ax, show_ylabel=False)
    return handles


def _bbox_overlap_area(first, second) -> float:
    intersection = mpl.transforms.Bbox.intersection(first, second)
    return 0.0 if intersection is None else float(intersection.width * intersection.height)


def _save_locator(out_dir: Path) -> tuple[Path, float]:
    fig = plt.figure(figsize=(6.40, 3.75), constrained_layout=False)
    map_ax = fig.add_axes([0.035, 0.035, 0.95, 0.94])
    _plot_locator(map_ax)
    fig.canvas.draw()
    path = out_dir / "Fig2a_locator.png"
    fig.savefig(path, dpi=400, facecolor="white")
    plt.close(fig)
    return path, 0.0


def _save_local_panel(
    out_dir: Path,
    filename: str,
    plotter,
    data: dict,
) -> tuple[Path, float]:
    # All local panels use the same canvas and axes-window geometry.
    fig, ax = plt.subplots(figsize=(4.35, 4.80), constrained_layout=False)
    fig.subplots_adjust(left=0.17, right=0.975, bottom=0.13, top=0.925)
    plotter(ax, data)
    fig.canvas.draw()
    path = out_dir / filename
    fig.savefig(path, dpi=400, facecolor="white")
    plt.close(fig)
    return path, 0.0


def _save_method_legend(out_dir: Path) -> tuple[Path, float]:
    # EM uses one shared colour across the three cases; map geometry distinguishes point
    # TEM, airborne AEM lines, and a TDEM loop without repeating legend rows.
    handles = [
        Line2D([], [], marker="s", linestyle="", color=PALETTE["tem"], markersize=4.5),
        Line2D([], [], marker="^", linestyle="", color=PALETTE["mt"], markersize=4.6),
        Line2D([], [], marker="o", linestyle="", color=PALETTE["gravity"], markersize=4.2),
        Line2D([], [], marker="o", linestyle="", color=PALETTE["sp"], markersize=3.5),
        Line2D([], [], color=PALETTE["ert"], lw=2.0),
        Line2D([], [], color=PALETTE["srt"], lw=1.5, ls=(0, (3, 1.5))),
        Line2D([], [], marker="*", linestyle="", color=PALETTE["well"], markeredgecolor="white", markeredgewidth=0.45, markersize=6.0),
        Line2D([], [], color=PALETTE["domain"], lw=1.15, ls=(0, (3, 2))),
        Line2D([], [], marker="o", linestyle="", markerfacecolor="none", markeredgecolor="#111111", markeredgewidth=1.0, markersize=5.0),
    ]
    labels = ["EM", "MT", "Gravity", "SP", "ERT", "SRT", "Deep well", "Model domain", "ERT-SRT-TEM column"]
    fig = plt.figure(figsize=(5.35, 1.55), constrained_layout=False)
    fig.legend(
        handles=handles,
        labels=labels,
        loc="center",
        bbox_to_anchor=(0.405, 0.5),
        ncol=2,
        frameon=False,
        columnspacing=2.3,
        handlelength=2.8,
        handletextpad=0.60,
        borderaxespad=0,
        fontsize=7.2,
    )

    # Minimal black-and-white cartographic north needle.
    compass = fig.add_axes([0.857, 0.215, 0.075, 0.570])
    compass.set_axis_off()
    compass.set_xlim(0, 1)
    compass.set_ylim(0, 1)
    ink = "#263238"
    cx, cy = 0.50, 0.45
    compass.text(cx, 0.965, "N", ha="center", va="top", fontsize=7.0, fontweight="bold", color=ink)
    compass.add_patch(
        Polygon(
            [[cx, 0.845], [0.405, cy], [0.595, cy]],
            closed=True, facecolor=ink, edgecolor=ink, linewidth=0.60, zorder=3,
        )
    )
    compass.add_patch(
        Polygon(
            [[cx, 0.105], [0.405, cy], [0.595, cy]],
            closed=True, facecolor="white", edgecolor=ink, linewidth=0.60, zorder=3,
        )
    )
    compass.add_patch(Circle((cx, cy), 0.025, facecolor="white", edgecolor=ink, linewidth=0.55, zorder=4))
    fig.canvas.draw()
    path = out_dir / "Fig2e_method_legend.png"
    fig.savefig(path, dpi=400, facecolor="white")
    plt.close(fig)
    return path, 0.0


def _qa_pngs(paths: list[Path], overlaps: dict[str, float]) -> dict:
    missing = [str(p) for p in paths if not p.exists() or p.stat().st_size == 0]
    if missing:
        raise AssertionError(f"Missing or empty PNG exports: {missing}")
    if any(area > 0 for area in overlaps.values()):
        raise AssertionError(f"Legend or scale-band overlap detected: {overlaps}")
    dimensions = {}
    for path in paths:
        with Image.open(path) as image:
            if image.format != "PNG":
                raise AssertionError(f"Unexpected format for {path.name}: {image.format}")
            min_height = 250 if path.name == "Fig2e_method_legend.png" else 1000
            if image.size[0] < 1600 or image.size[1] < min_height:
                raise AssertionError(f"PNG resolution too small for {path.name}: {image.size}")
            dimensions[path.name] = list(image.size)
    return {
        "files": len(paths),
        "format": "PNG only",
        "dpi": 400,
        "dimensions_px": dimensions,
        "legend_overlap_px2": overlaps,
    }


def build_figure(root: str | Path | None = None) -> dict:
    """Export four map panels plus a shared method legend and run collision QA."""
    _set_style()
    root = _find_root(Path(root) if root is not None else None)
    out_dir = root / "examples" / "Figure" / "fig2"
    out_dir.mkdir(parents=True, exist_ok=True)
    d = _load_data(root)

    outputs: list[Path] = []
    overlaps: dict[str, float] = {}
    path, overlap = _save_locator(out_dir)
    outputs.append(path)
    overlaps[path.name] = overlap

    local_specs = [
        ("Fig2b_utah_forge.png", _plot_utah, d["utah"]),
        ("Fig2c_cedar_rapids.png", _plot_cedar, d["cedar"]),
        ("Fig2d_llano_uplift.png", _plot_llano, d["llano"]),
    ]
    for filename, plotter, case_data in local_specs:
        path, overlap = _save_local_panel(out_dir, filename, plotter, case_data)
        outputs.append(path)
        overlaps[path.name] = overlap

    path, overlap = _save_method_legend(out_dir)
    outputs.append(path)
    overlaps[path.name] = overlap

    qa = _qa_pngs(outputs, overlaps)
    return {
        "output_dir": str(out_dir),
        "pngs": [str(path) for path in outputs],
        "qa": qa,
        "figure_contract": {
            "claim": "Four independent maps introduce location, scale, survey coverage, and model domains.",
            "assembly": "Independent PNG maps plus one shared method-only legend for manual PowerPoint assembly.",
            "legend_policy": "Panels b-d omit legends; panel e provides the shared method-and-domain legend.",
        },
    }

## Export study-site panels

In [ ]:
result = build_figure()
print(json.dumps(result['figure_contract'], indent=2))
print(json.dumps(result['qa'], indent=2))


## Export check

Every listed panel must exist and have nonzero dimensions.

In [ ]:
from PIL import Image
OUT = _find_root(None) / 'examples' / 'Figure' / 'fig2'
expected = ['Fig2a_locator.png', 'Fig2b_utah_forge.png', 'Fig2c_cedar_rapids.png', 'Fig2d_llano_uplift.png', 'Fig2e_method_legend.png']
for name in expected:
    path = OUT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as panel:
        assert panel.width > 0 and panel.height > 0, path
print(f'Figure panels checked: {len(expected)} in {OUT}')
